In [ ]:
import pyspark
import dxpy
import dxdata
import json
import numpy as np
from bokeh.io import show, output_notebook
from bokeh.layouts import gridplot
import random
output_notebook()

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

In [ ]:
db_name = "arb_db"
db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database")['id']

In [ ]:
import hail as hl
hl.init(sc=sc, default_reference='GRCh38')

In [ ]:
tb_name = "bp-for-export.ht"
url = f"dnax://{db_uri}/{tb_name}"
    
bp_meas = hl.read_table(url)

In [ ]:
participant_list = list(set(bp_meas.eid.collect()))

In [ ]:
len(participant_list)

In [ ]:
kinship = hl.import_table(
    '<PATH_TO_UKB_REL_DAT>',
    delimiter=' ',
    types={
        'HetHet': hl.tfloat32,
        'IBS0': hl.tfloat32,
        'Kinship': hl.tfloat32,
    }
).persist()
kinship.count()

def remove_rel(participant_list, kinship, kinship_th=0.0884):
    participant_list_hl = hl.literal(set(participant_list))
    
    kinship = kinship.filter(kinship.Kinship > kinship_th)
    kinship = kinship.filter(
        participant_list_hl.contains(kinship.ID1)
        & participant_list_hl.contains(kinship.ID2)
    )

    kinship = kinship.annotate(
        i=hl.struct(
            id=kinship.ID1,
        ),
        j=hl.struct(
            id=kinship.ID2,
        ),
    ).persist()
    
    close_rel_ids = (
        kinship.aggregate(hl.agg.collect_as_set(kinship.ID1))
        |  kinship.aggregate(hl.agg.collect_as_set(kinship.ID2))
    )
    print(
        f'Number of closely related individuals (kinship > '
        f'{kinship_th}): {len(close_rel_ids)}'
    )
    
    mis_remove = hl.maximal_independent_set(
        kinship.i, kinship.j, keep=False
    ).persist()
    eids_remove = set(mis_remove.node.id.collect())
    print(f'Removed: {len(eids_remove)}')

    return set(participant_list) - eids_remove

In [ ]:
x = remove_rel(participant_list, kinship, kinship_th=0.0884)

In [ ]:
len(x)

In [ ]:
import pandas as pd

In [ ]:
pd.Series(list(x)).to_csv('up-to-third-degree.tsv', index = False, sep = "\t")

In [ ]:
x = remove_rel(participant_list, kinship, kinship_th=0.177)

In [ ]:
len(x)

In [ ]:
pd.Series(list(x)).to_csv('up-to-second-degree.tsv', index = False, sep = "\t")